# 01 — Preprocessing & Sampling

Builds three stratified samples (debug / classical / neural) from the raw corpus, applies light text cleaning, and writes parquet artifacts that downstream notebooks consume.

Sample sizes are configurable below. Defaults are conservative for a free-tier laptop / Colab session.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import re
import polars as pl
import pandas as pd
from sklearn.model_selection import train_test_split

from utils import ARTIFACTS_DIR, RANDOM_STATE, find_amazon_csv, set_seed

set_seed()

# Sample sizes — shrink for faster iteration
N_DEBUG    = 100_000   # quick lexicon + classical demo
N_CLASSICAL = 500_000  # full classical-ML reporting
N_NEURAL   = 200_000   # transformer fine-tuning

In [ ]:
csv_path = find_amazon_csv()
df = pl.read_csv(csv_path, has_header=False, new_columns=["polarity", "title", "text"])
df = df.drop_nulls(subset=["polarity", "text"])
print("Loaded:", df.shape)

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")

def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = s.lower()
    s = URL_RE.sub(" ", s)
    s = s.replace("&quot;", '"').replace("&amp;", "&")
    s = WS_RE.sub(" ", s).strip()
    return s

# Pull title + text into a single 'review' field (matches reference repo style)
df = df.with_columns([
    (pl.col("title").fill_null("") + ". " + pl.col("text").fill_null("")).alias("raw")
])
df = df.with_columns(
    pl.col("raw").map_elements(clean_text, return_dtype=pl.Utf8).alias("review")
)
df = df.with_columns((pl.col("polarity") - 1).alias("label"))  # {0, 1}
print(df.select(["polarity", "label", "review"]).head(3))

In [ ]:
def stratified_sample(df_pl: pl.DataFrame, n: int, seed: int = RANDOM_STATE) -> pl.DataFrame:
    pdf = df_pl.to_pandas()
    sampled, _ = train_test_split(
        pdf, train_size=n, stratify=pdf["label"], random_state=seed
    )
    return pl.from_pandas(sampled.reset_index(drop=True))

# Build the three samples; write each as parquet
samples = {
    "debug":     stratified_sample(df, N_DEBUG),
    "classical": stratified_sample(df, N_CLASSICAL),
    "neural":    stratified_sample(df, N_NEURAL),
}
for name, sample in samples.items():
    out = ARTIFACTS_DIR / f"sample_{name}.parquet"
    sample.write_parquet(out)
    print(f"  {name:9s} -> {out.name}  ({sample.shape[0]:>8,} rows)")

In [ ]:
# Train / val / test split on the *neural* sample (used by every later notebook).
neural = samples["neural"].to_pandas()
train, temp = train_test_split(
    neural, test_size=0.20, stratify=neural["label"], random_state=RANDOM_STATE
)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp["label"], random_state=RANDOM_STATE
)
for name, split in [("train", train), ("val", val), ("test", test)]:
    out = ARTIFACTS_DIR / f"split_{name}.parquet"
    pl.from_pandas(split.reset_index(drop=True)).write_parquet(out)
    print(f"  {name:5s} -> {out.name}  ({len(split):>7,} rows)")

## Artifacts produced

- `sample_debug.parquet` — 100K stratified
- `sample_classical.parquet` — 500K stratified
- `sample_neural.parquet` — 200K stratified
- `split_train.parquet`, `split_val.parquet`, `split_test.parquet` — 80/10/10 split of `sample_neural`

Downstream notebooks default to `split_*.parquet`. Swap to the larger `sample_classical.parquet` for the final classical-ML reporting run.